# Module 1: Data Ingestion & Preprocessing
This notebook correctly merges the MPLADS datasets by extracting the unique Project ID instead of concatenating them.

In [1]:
import pandas as pd
import numpy as np
import re
import os


In [2]:
# Load datasets
data_dir = '../data'
rec_df = pd.read_csv(os.path.join(data_dir, 'Works Recommended.csv'))
sanc_df = pd.read_csv(os.path.join(data_dir, 'Works Sanctioned.csv'))
exp_df = pd.read_csv(os.path.join(data_dir, 'Expenditure on Completed and On-going Works as on Date.csv'))
comp_df = pd.read_csv(os.path.join(data_dir, 'Works Completed.csv'))

print(f"Recommended: {len(rec_df)}")
print(f"Sanctioned: {len(sanc_df)}")
print(f"Expenditure: {len(exp_df)}")
print(f"Completed: {len(comp_df)}")

/var/folders/1f/slhttrhj3rq1pf636jk7t00w0000gn/T/ipykernel_69107/2068429900.py:3: DtypeWarning: Columns (0,9) have mixed types. Specify dtype option on import or set low_memory=False.
  rec_df = pd.read_csv(os.path.join(data_dir, 'Works Recommended.csv'))
/var/folders/1f/slhttrhj3rq1pf636jk7t00w0000gn/T/ipykernel_69107/2068429900.py:4: DtypeWarning: Columns (0,10) have mixed types. Specify dtype option on import or set low_memory=False.
  sanc_df = pd.read_csv(os.path.join(data_dir, 'Works Sanctioned.csv'))


Recommended: 102711
Sanctioned: 77995
Expenditure: 82367
Completed: 33842


/var/folders/1f/slhttrhj3rq1pf636jk7t00w0000gn/T/ipykernel_69107/2068429900.py:5: DtypeWarning: Columns (0,10) have mixed types. Specify dtype option on import or set low_memory=False.
  exp_df = pd.read_csv(os.path.join(data_dir, 'Expenditure on Completed and On-going Works as on Date.csv'))


In [3]:
# Function to extract the 6-digit project ID from the Work string
def extract_id(text):
    if pd.isna(text):
        return np.nan
    # Look for a 5 or 6 digit number that comes after a slash and before a hyphen
    match = re.search(r'/(\d{5,7})(?:-|$)', str(text))

    if match:
        return match.group(1)
    # If the text itself is just a number (like in Expenditure)
    if str(text).strip().isdigit():
        return str(text).strip()
    return np.nan

In [4]:
# Apply the extraction function to create a unified primary key
rec_df['Project_ID'] = rec_df['WORK'].apply(extract_id)
sanc_df['Project_ID'] = sanc_df['Work'].apply(extract_id)
exp_df['Project_ID'] = exp_df['Work ID'].apply(extract_id)
comp_df['Project_ID'] = comp_df['Work'].apply(extract_id)

print(f"Valid IDs extracted - Rec: {rec_df['Project_ID'].notna().sum()}, Sanc: {sanc_df['Project_ID'].notna().sum()}, Exp: {exp_df['Project_ID'].notna().sum()}, Comp: {comp_df['Project_ID'].notna().sum()}")

Valid IDs extracted - Rec: 77636, Sanc: 77994, Exp: 82366, Comp: 33841


In [5]:
# Clean and normalize text columns
for df in [rec_df, sanc_df]:
    if 'Work description' in df.columns:
        df['Work description'] = df['Work description'].str.lower().str.strip()
    if 'Work category' in df.columns:
        df['Work category'] = df['Work category'].str.lower().str.strip()

In [6]:
# Parse dates
date_cols = ['Recommended date', 'Sanction Date', 'Expenditure Date', 'Completion Date']
for df in [rec_df, sanc_df, exp_df, comp_df]:
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

In [7]:
# Drop unnecessary columns before merging to avoid duplication
# We will use the Sanctioned DF as the anchor for description, so we just take amounts from others
rec_subset = rec_df[['Project_ID', 'Work description', 'Work category', 'Hon\'ble Members of Parliament', 'Constituency', 'State', 'Recommended date', 'RECOMMENDED AMOUNT   ( ₹ )']].drop_duplicates(subset=['Project_ID'])
sanc_subset = sanc_df[['Project_ID', 'Sanction Date', 'Sanction Amount ( ₹ )', 'Work Status']].drop_duplicates(subset=['Project_ID'])

# Expenditure might have multiple payments per project, so we group by Project_ID and sum the disbursed amount
# First, clean the amount column to be numeric
exp_df['Fund Disbursed Amount ( ₹ )'] = pd.to_numeric(exp_df['Fund Disbursed Amount ( ₹ )'].astype(str).str.replace(',', ''), errors='coerce')
exp_grouped = exp_df.groupby('Project_ID', as_index=False).agg({
    'Fund Disbursed Amount ( ₹ )': 'sum',
    'Payment Status': 'last'
})

comp_subset = comp_df[['Project_ID', 'Completion Date', 'Amount Disbursed ( ₹ )']].drop_duplicates(subset=['Project_ID'])

In [8]:
# Merge datasets horizontally on Project_ID
# We use outer joins to ensure we don't lose any projects
master_df = rec_subset.merge(sanc_subset, on='Project_ID', how='outer')
master_df = master_df.merge(exp_grouped, on='Project_ID', how='outer')
master_df = master_df.merge(comp_subset, on='Project_ID', how='outer')

print(f"Master dataset created with {len(master_df)} rows and {len(master_df.columns)} columns.")

Master dataset created with 77995 rows and 15 columns.


In [9]:
# Show how beautifully the lifecycle tracks now!
master_df[['Project_ID', 'RECOMMENDED AMOUNT   ( ₹ )', 'Sanction Amount ( ₹ )', 'Fund Disbursed Amount ( ₹ )', 'Work Status']].head(10)

,Project_ID,RECOMMENDED AMOUNT ( ₹ ),Sanction Amount ( ₹ ),Fund Disbursed Amount ( ₹ ),Work Status
0,133166,497185.0,497185.0,497185.0,Physical Inspection
1,133167,500000.0,500000.0,NaN,Sanction
2,133190,450000.0,450000.0,NaN,Sanction
3,133191,1500000.0,1500000.0,850909.0,Work partially Completed
4,133301,398009.0,398009.0,398009.0,Physical Inspection
5,133303,399104.0,399104.0,399104.0,Physical Inspection
6,133304,396154.0,396154.0,396154.0,Physical Inspection
7,133305,400000.0,400000.0,NaN,Sanction
8,133306,495627.0,495627.0,495627.0,Physical Inspection
9,133307,398955.0,398955.0,398955.0,Physical Inspection


In [10]:
# Save the merged master dataset for the next modules
master_df.to_csv('master_mplads_data.csv', index=False)
print("Saved to master_mplads_data.csv")

Saved to master_mplads_data.csv


In [11]:
master_df.head()


,Project_ID,Work description,Work category,Hon'ble Members of Parliament,Constituency,State,Recommended date,RECOMMENDED AMOUNT ( ₹ ),Sanction Date,Sanction Amount ( ₹ ),Work Status,Fund Disbursed Amount ( ₹ ),Payment Status,Completion Date,Amount Disbursed ( ₹ )
0,133166,construction of community bhavan at navalgund ...,normal/others,Pralhad Venkatesh Joshi,DHARWAD,Karnataka,2024-07-08,497185.0,2024-07-09,497185.0,Physical Inspection,497185.0,Payment Success,2024-10-14,497185
1,133167,construction of college room of cbs charitabl...,trust and society,Pralhad Venkatesh Joshi,DHARWAD,Karnataka,2024-07-08,500000.0,2025-09-18,500000.0,Sanction,NaN,NaN,NaT,NaN
2,133190,construction of community bhavan of veerashaiv...,trust and society,Pralhad Venkatesh Joshi,DHARWAD,Karnataka,2024-07-08,450000.0,2024-09-23,450000.0,Sanction,NaN,NaN,NaT,NaN
3,133191,construction of maremma cultural bhavan at nee...,normal/others,Pralhad Venkatesh Joshi,DHARWAD,Karnataka,2024-07-08,1500000.0,2025-05-29,1500000.0,Work partially Completed,850909.0,Payment Success,NaT,NaN
4,133301,construction of cultural bhavan at kundagol tq...,normal/others,Pralhad Venkatesh Joshi,DHARWAD,Karnataka,2024-07-09,398009.0,2024-08-22,398009.0,Physical Inspection,398009.0,Payment Success,2025-02-03,398009
